# Chapter 05：Reduction Sum / Max

**目标**：对二维 tensor `[M, N]` 做 row-wise sum 和 max。核心概念是一行一个 program，以及用 reduction 合并一个 offset 向量。

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name.startswith("chapter_"):
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
import triton
import triton.language as tl

from common.benchmark import bench
from common.check import assert_close
from common.utils import get_device, set_seed

device = get_device()
set_seed(0)

## 1. PyTorch reference

`N=513` 不是 2 的幂，便于验证 power-of-two block 和 mask。

In [ ]:
M, N = 1024, 513
x = torch.randn(M, N, device=device)
sum_ref = torch.sum(x, dim=1)
max_ref = torch.max(x, dim=1).values

## 2. BLOCK_SIZE 与 N

Triton reduction 常使用 2 的幂长度。wrapper 选择 `next_power_of_2(N)`；超出 N 的 lane 对 sum 使用 0，对 max 使用负无穷，这些是各自的中性值。

In [ ]:
print(f"N={N}, BLOCK_SIZE={triton.next_power_of_2(N)}")

## 3. Row sum 与 row max kernel

In [ ]:
@triton.jit
def row_sum_kernel(x_ptr, output_ptr, n_cols, BLOCK_SIZE: tl.constexpr):
    row = tl.program_id(0)
    cols = tl.arange(0, BLOCK_SIZE)
    mask = cols < n_cols
    values = tl.load(x_ptr + row * n_cols + cols, mask=mask, other=0.0)
    tl.store(output_ptr + row, tl.sum(values, axis=0))

@triton.jit
def row_max_kernel(x_ptr, output_ptr, n_cols, BLOCK_SIZE: tl.constexpr):
    row = tl.program_id(0)
    cols = tl.arange(0, BLOCK_SIZE)
    mask = cols < n_cols
    values = tl.load(x_ptr + row * n_cols + cols, mask=mask, other=-float("inf"))
    tl.store(output_ptr + row, tl.max(values, axis=0))

## 4. Wrapper functions

grid 是 `(M,)`，因此第 `row` 个 program 只处理第 `row` 行。输入要求 contiguous，地址可以写成 `row * N + cols`。

In [ ]:
def _reduce(kernel, x):
    if x.ndim != 2 or not x.is_cuda or not x.is_contiguous():
        raise ValueError("expected a contiguous 2D CUDA tensor")
    M, N = x.shape
    if N == 0:
        raise ValueError("N must be positive")
    output = torch.empty(M, device=x.device, dtype=x.dtype)
    if M > 0:
        block_size = triton.next_power_of_2(N)
        num_warps = 8 if block_size >= 2048 else 4
        kernel[(M,)](x, output, N, BLOCK_SIZE=block_size, num_warps=num_warps)
    return output

def row_sum(x):
    return _reduce(row_sum_kernel, x)

def row_max(x):
    return _reduce(row_max_kernel, x)

## 5. Correctness check

In [ ]:
assert_close("row sum", row_sum(x), sum_ref, rtol=1e-3, atol=1e-3)
assert_close("row max", row_max(x), max_ref)

## 6. Benchmark 不同 N

In [ ]:
for n_cols in (127, 513, 1024):
    sample = torch.randn(1024, n_cols, device=device)
    print(
        f"N={n_cols:4d} sum: torch={bench(lambda: torch.sum(sample, dim=1)):.3f} ms, "
        f"triton={bench(lambda: row_sum(sample)):.3f} ms"
    )
    print(
        f"N={n_cols:4d} max: torch={bench(lambda: torch.max(sample, dim=1).values):.3f} ms, "
        f"triton={bench(lambda: row_max(sample)):.3f} ms"
    )

## 小结与练习

mask 的 `other` 必须选择不会改变 reduction 结果的中性值。

**练习**：把 `N` 改为 1000，确认 `BLOCK_SIZE` 是 1024，并重新检查 sum/max。